## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-10-18-54-42 +0000,nypost,Knife-wielding Central Park rape suspect buste...,https://nypost.com/2026/11/10/us-news/central-...
1,2026-11-10-18-51-37 +0000,nypost,Illegal criminal alien’s A-list celeb links re...,https://nypost.com/2026/11/10/us-news/iranian-...
2,2026-11-10-18-51-34 +0000,nypost,The unsettling reality that the Nvidia spy sca...,https://nypost.com/2026/11/10/us-news/the-unse...
3,2026-11-10-18-49-21 +0000,nyt,"In Interview, Sonia Gandhi Recalls Love and Lo...",https://www.nytimes.com/2026/11/10/world/asia/...
4,2026-11-10-18-48-52 +0000,nyt,"Who Is Alice Huang, the Woman in the Middle of...",https://www.nytimes.com/2026/11/10/technology/...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2477/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
435,trump,46
28,nvidia,38
42,huang,26
188,iran,24
779,jensen,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
122,2026-11-10-15-45-06 +0000,nyt,Nvidia CEO Jensen Huang Told Trump He Did Not ...,https://www.nytimes.com/2026/11/09/us/politics...,225
375,2026-11-09-22-16-39 +0000,nypost,Nvidia boss Jensen Huang had ‘absolutely no id...,https://nypost.com/2026/11/09/us-news/nvidia-b...,214
197,2026-11-10-11-59-50 +0000,nypost,FBI is ‘looking into’ Nvidia boss Jensen Huang...,https://nypost.com/2026/11/10/us-news/fbi-is-l...,199
353,2026-11-09-23-10-47 +0000,wapo,Trump says Nvidia CEO denied knowing woman he ...,https://www.washingtonpost.com/technology/2026...,187
423,2026-11-09-20-39-13 +0000,nypost,Trader warned of Nvidia CEO Jensen Huang’s ‘un...,https://nypost.com/2026/11/09/business/trader-...,184


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
122,225,2026-11-10-15-45-06 +0000,nyt,Nvidia CEO Jensen Huang Told Trump He Did Not ...,https://www.nytimes.com/2026/11/09/us/politics...
120,107,2026-11-10-15-47-00 +0000,wsj,Defense Secretary Pete Hegseth gave a Pentagon...,https://www.wsj.com/politics/national-security...
384,72,2026-11-09-22-06-00 +0000,wsj,Veteran GOP lawmaker Steve Scalise is going he...,https://www.wsj.com/politics/why-steve-scalise...
54,70,2026-11-10-17-36-51 +0000,nypost,Gas prices drop below $4 a gallon for first ti...,https://nypost.com/2026/11/10/business/gas-pri...
309,68,2026-11-10-02-00-00 +0000,nypost,Ex-Dem Senate leader asks California Supreme C...,https://nypost.com/2026/11/09/us-news/ex-senat...
153,62,2026-11-10-14-20-09 +0000,cbc,Anthropic eyes $1 trillion US valuation in hot...,https://www.cbc.ca/news/business/anthropic-ipo...
383,62,2026-11-09-22-06-57 +0000,nypost,Trump holds White House briefing with UFC’s Da...,https://nypost.com/2026/11/09/us-news/trump-ho...
230,58,2026-11-10-10-48-00 +0000,nyt,Here’s why last year’s government shutdown wil...,https://www.nytimes.com/live/2026/11/10/busine...
405,50,2026-11-09-21-22-11 +0000,wapo,U.S. gives Russia an early win before Ukraine ...,https://www.washingtonpost.com/world/2026/11/0...
300,49,2026-11-10-03-28-00 +0000,wsj,California voters rejected a proposed billiona...,https://www.wsj.com/politics/elections/califor...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
